<a href="https://colab.research.google.com/github/lucianoselimaj/SigExt/blob/MMD/notebooks/SigExt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#@title requirements
!pip install transformers datasets accelerate evaluate rouge_score torch jsonlines rake_nltk pytorch_lightning
!pip install rapidfuzz
!pip install boto3
!pip install mistralai
!pip install --upgrade mistralai

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 23.2 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=ccf824063b5a114a78b829e791da449c5478dd272f71ac350ca8fd1e41cc56bd
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.1/49.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Clone the Repo

In [3]:
# @markdown check the check-box to clone the repo from sorce, <b>otherwise it will be loaded from shared google drive files</b>

clone_repo = False # @param {type:"boolean"}
if clone_repo:
  !git clone https://github.com/lucianoselimaj/SigExt.git
  path = "/SigExt"

else:
  path = "/content/drive/MyDrive/DNLP-storage/SigExt"


# Preparing Data

In [4]:
# @title Download requirements for data preparing

%cd {path}

import nltk
nltk.download('stopwords')
nltk.download('punkt_tab')

/content/drive/MyDrive/DNLP-storage/SigExt


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [5]:
# @title Run the `prepare_data.py` script
# @markdown Here you can select the dataset you want to use for the training. </br> If you want to use a new dataset, make sure to also change the `path_to_dataset`. </br> No new file will be downloaded or processed if `path_to_dataset` already exists!!

import os
dataset = "cnn" #@param{type:"string"}
path_to_dataset = "experiments/cnn_dataset/" #@param{type:"string"}
if not os.path.exists(path+"/"+path_to_dataset):
  print('New dataset...')
  !python3 src/prepare_data.py --dataset {dataset} --output_dir {path_to_dataset}
  pass

else:
  print(f"Directory already exist. `{path_to_dataset}` will be used")

Directory already exist. `experiments/cnn_dataset/` will be used


# Train

You can modify path to check-points and outputs to load or save different files.
**To keep track of procedure and training routh, insert paths you create in this text down below, under the `paths` ( DO NOT FORGET THE DESCRIPTION :) thank you in advance.  )**



 **⚠️Even for test runs, Change two paths to avoid overwriting existing results⚠️**


---


**Paths**
*   `experiments/cnn_extractor_model/` Baseline code (dataset: cnn),  no modification. output in `experiments/cnn_dataset_with_keyphrase/`
*   `new_path` add you new path here...





In [6]:

path_to_checkpoint = "experiments/cnn_extractor_model/" #@param{type:"string"}
path_to_output = "experiments/cnn_dataset_with_keyphrase/" #@param{type:"string"}

train_longformer_extractor = False #@param{type:"boolean"}
inference_longformer_extractor = False #@param{type:"boolean"}


In [7]:
#@title Calling `train_longformer_extractor_context.py` to train the longformer
if train_longformer_extractor:
  !python3 src/train_longformer_extractor_context.py \
    --dataset_dir {path_to_dataset} \
    --checkpoint_dir {path_to_checkpoint}

In [8]:
# #@title Calling `inference_longformer_extractor.py`
if inference_longformer_extractor:
  !python3 src/inference_longformer_extractor.py \
    --dataset_dir {path_to_dataset} \
    --checkpoint_dir {path_to_checkpoint} \
    --output_dir {path_to_output}

In [17]:
summerizing_result_path = "experiments/cnn_extsig_predictions_mistral-small-2603_k15/" #@param{type:"string"}
top_k_kp = 15 #@param{type:"integer"}

In [ ]:
import sys
import os
from types import ModuleType
import time
def chat(prompt,client):
  chat_response = client.chat.complete(
            model="mistral-small-2603",
            messages=[{"role": "user", "content": prompt["prompt_input"]}]
        )
  return chat_response.choices[0].message.content

def predict_one_eg_mistral(prompt):
    from mistralai.client import Mistral
    client = Mistral(api_key="YdbLhqEepgXav0x8ruTJiWR20GsgWCrD")
    err = None
    time.sleep(1.1)
    for i in range(5):
      try:
        chat_response = chat(prompt,client)
        return chat_response

      except Exception as e:

        if "429" in str(e):
          print("Rate limit hit! Sleeping for 10 seconds...")
          time.sleep(10)
          err = e

    return f"Error: {err}"

# We replace the multiprocessing Pool with a "fake" Pool that doesn't use multiple processes
class FakePool:
    def __init__(self, processes=None): pass
    def __enter__(self): return self
    def __exit__(self, *args): pass
    def imap(self, func, iterable):
        # Instead of multiprocessing, we just use a standard map
        return map(func, iterable)

import multiprocessing
multiprocessing.Pool = FakePool

m = ModuleType("bedrock_utils")
m.predict_one_eg_mistral = predict_one_eg_mistral
m.predict_one_eg_claude_instant = lambda x: "Not configured"
sys.modules["bedrock_utils"] = m

sys.argv = [
    "zs_summarization.py",
    "--model_name", "mistral",
    "--kw_strategy", "sigext_topk",
    "--kw_model_top_k", str(top_k_kp),
    "--dataset", dataset,
    "--dataset_dir", path_to_output,
    "--output_dir", summerizing_result_path
]


sys.path.append('/content/drive/MyDrive/DNLP-storage/SigExt/src')
import zs_summarization

import importlib
importlib.reload(zs_summarization)

zs_summarization.main()

  2%|▏         | 10/500 [00:23<18:51,  2.31s/it]